# Data cleaning & Preprocessing

> TODO adicionar descrição breve sobre o que fiz

## Data cleaning

In [195]:
import pandas as pd 
import numpy as np 
import seaborn as sns
import missingno as msno
sns.set_theme(style='darkgrid')
import matplotlib.pyplot as plt

In [196]:
data = pd.read_excel('Assignment/DataSets/UCMF.xls')

data = data.rename(columns={
    'IDADE': 'Idade',
    'PULSOS':'Pulsos',
    'PA SISTOLICA': 'SBP',
    'PA DIASTOLICA': 'DBP',
    'PPA': 'Result SBP-DBP',
    'NORMAL X ANORMAL': 'Patologia',
    'SOPRO' : 'Sopro',
    'SEXO':'Sexo',
    'HDA 1': 'HDA1',
    'MOTIVO1' : 'Motivo1',
    'MOTIVO2': 'Motivo2'
})

In [197]:
data.columns

Index(['ID', 'Peso', 'Altura', 'IMC', 'Atendimento', 'DN', 'Idade', 'Convenio',
       'Pulsos', 'SBP', 'DBP', 'Result SBP-DBP', 'Patologia', 'B2', 'Sopro',
       'FC', 'HDA1', 'HDA2', 'Sexo', 'Motivo1', 'Motivo2'],
      dtype='object')

### Remove Irrelevant features

> Removed Cardiac frequency (after consulting someone that understands this matter) and they said that this feature cannot be 'trusted' to decide if children have or dont have a cardiac pathology.

In [198]:
data = data.drop(['ID','Atendimento','DN','FC','Convenio'],axis=1)

In [199]:
data.columns

Index(['Peso', 'Altura', 'IMC', 'Idade', 'Pulsos', 'SBP', 'DBP',
       'Result SBP-DBP', 'Patologia', 'B2', 'Sopro', 'HDA1', 'HDA2', 'Sexo',
       'Motivo1', 'Motivo2'],
      dtype='object')

### Filter records with desired target group age (2-19)

In [200]:
data.shape

(17873, 16)

In [201]:

data = data[data['Idade'] != '#!VALUE!']
data = data[data['Idade'] != '#VALUE!']

In [202]:
data.shape

(17753, 16)

In [203]:
data['Idade'] = data['Idade'].astype(float)


In [204]:
data['Idade'] = data['Idade'].abs()

In [205]:
data = data[data['Idade'].between(2,19)]
data.shape

(12384, 16)

In [206]:
data['Idade'].value_counts

<bound method IndexOpsMixin.value_counts of 2         4.05
4         9.60
5         4.40
6        12.89
7         5.89
         ...  
17862    12.30
17863     8.30
17865     8.67
17867     2.14
17872    11.38
Name: Idade, Length: 12384, dtype: float64>

### Handle Inconsistent values

In [207]:
# Patologia
print("(Before)", data['Patologia'].value_counts())
map = {
    'anormal': 'Anormal',
    'Normais': 'Normal'
}
data['Patologia'] = data['Patologia'].replace(map)
print("(After)", data['Patologia'].value_counts())



(Before) Patologia
Normal     7768
Anormal    4315
anormal       1
Normais       1
Name: count, dtype: int64
(After) Patologia
Normal     7769
Anormal    4316
Name: count, dtype: int64


In [208]:
# Sexo
print("(Before)",data['Sexo'].value_counts())
data['Sexo'] = data['Sexo'].str.strip().str.capitalize()
map = {
    'M': 'Masculino',
    'F': 'Feminino'
}
data['Sexo'] = data['Sexo'].replace(map)
print("(After)", data['Sexo'].value_counts())

(Before) Sexo
M                6647
F                4884
Masculino         463
Feminino          178
Indeterminado     147
masculino          62
Name: count, dtype: int64
(After) Sexo
Masculino        7172
Feminino         5062
Indeterminado     147
Name: count, dtype: int64


In [209]:
# Pulsos
print("(Before)",data['Pulsos'].value_counts())
data['Pulsos'] = data['Pulsos'].str.strip().str.capitalize()
map = {
    'Normais': 'Normal',
    'NORMAIS': 'Normal',
    'Diminuídos': 'Femorais diminuidos',
}
data['Pulsos'] = data['Pulsos'].replace(map)
print("(After)", data['Pulsos'].value_counts())


(Before) Pulsos
Normais                12019
Outro                     26
Amplos                    17
Femorais diminuidos        9
Diminuídos                 7
NORMAIS                    1
Name: count, dtype: int64
(After) Pulsos
Normal                 12020
Outro                     26
Amplos                    17
Femorais diminuidos       16
Name: count, dtype: int64


In [210]:
# B2
print("(Before)",data['B2'].value_counts())
data['B2'] = data['B2'].str.strip().str.capitalize()
map = {
    'Única': 'Unico',
    'Desdob fixo': 'Split fixo'
}
data['B2'] = data['B2'].replace(map)
print("(After)", data['B2'].value_counts())

(Before) B2
Normal           11729
Hiperfonética      136
Desdob fixo        115
Outro               79
Única               28
Name: count, dtype: int64
(After) B2
Normal           11729
Hiperfonética      136
Split fixo         115
Outro               79
Unico               28
Name: count, dtype: int64


In [211]:
# Sopro
print("(Before)",data['Sopro'].value_counts())
data['Sopro'] = data['Sopro'].str.strip().str.capitalize()
map = {
    'ausente': 'Ausente',
    'sistólico': 'Sistólico',
    'contínuo': 'Contínuo',
    'diastólico': 'Diastólico'
}
data['Sopro'] = data['Sopro'].replace(map)
print("(After)", data['Sopro'].value_counts())

(Before) Sopro
ausente                   8258
Sistólico                 3023
sistólico                  772
contínuo                    16
Contínuo                    13
diastólico                   8
Sistolico e diastólico       2
Name: count, dtype: int64
(After) Sopro
Ausente                   8258
Sistólico                 3795
Contínuo                    29
Diastólico                   8
Sistolico e diastólico       2
Name: count, dtype: int64


In [212]:
# Motivo1
data['Motivo1'].value_counts()
print("(Before)",data['Motivo1'].value_counts())
map = {
    '5 - Parecer cardiológico' : 'Triagem Cardiológica',
    '6 - Suspeita de cardiopatia': 'Possivel cardiopatia',
    '2 - Check-up' : 'Checkup de rotina',
    '1 - Cardiopatia já estabelecida': 'Cardiopatia',
    '7 - Outro' : 'Outros'
}
data['Motivo1'] = data['Motivo1'].replace(map)
print("(After)", data['Motivo1'].value_counts())

(Before) Motivo1
5 - Parecer cardiológico           6472
6 - Suspeita de cardiopatia        3917
2 - Check-up                        789
1 - Cardiopatia já estabelecida     784
7 - Outro                           314
Name: count, dtype: int64
(After) Motivo1
Triagem Cardiológica    6472
Possivel cardiopatia    3917
Checkup de rotina        789
Cardiopatia              784
Outros                   314
Name: count, dtype: int64


In [213]:
# Motivo2
data['Motivo2'].value_counts()
print("(Before)",data['Motivo2'].value_counts())
map = {
    '5 - Cirurgia': 'Cirurgia',
    '6 - Sopro' : 'Presença de Sopro',
    '5 - Atividade física' : 'Atividade física',
    'Outro' : 'Outros',
    '1 - Cardiopatia congenica' : 'Cardiopatia congenica',
    '6 - Dor precordial' : 'Outros',
    '6 - Palpitação/taquicardia/arritmia' : 'Outros',
    '6 - HAS/dislipidemia/obesidade' : 'Fatores de risco',
    '6 - Dispnéia' : 'Outros',
    '1 - Cardiopatia adquirida' :'Outros',
    '6 - Cianose' : 'Outros',
    '6 - Cardiopatia na familia' : 'Fatores de risco',
    '6 - Cansaço' : 'Outros',
    '6 - Alterações de pulso/perfusão' : 'Outros',
    '6 - Cianose e dispnéia' : 'Outros',
    '5 - Uso de cisaprida' : 'Outros'

}
data['Motivo2'] = data['Motivo2'].replace(map)
print("(After)", data['Motivo2'].value_counts())


(Before) Motivo2
5 - Cirurgia                           3352
6 - Sopro                              1721
5 - Atividade física                   1069
Outro                                   720
1 - Cardiopatia congenica               639
6 - Dor precordial                      616
6 - Palpitação/taquicardia/arritmia     452
6 - HAS/dislipidemia/obesidade          396
6 - Dispnéia                            240
1 - Cardiopatia adquirida               112
6 - Cianose                              51
6 - Cardiopatia na familia               31
6 - Cansaço                              16
6 - Alterações de pulso/perfusão          2
6 - Cianose e dispnéia                    2
5 - Uso de cisaprida                      1
Name: count, dtype: int64
(After) Motivo2
Cirurgia                 3352
Outros                   2212
Presença de Sopro        1721
Atividade física         1069
Cardiopatia congenica     639
Fatores de risco          427
Name: count, dtype: int64


### Remove incorect/impossible/irrelevant values

In [214]:
# Remover objectos com valores incorrectos de SBP (da-mos uma 'folga' de 10 para
# limite superior e inferior)
import json
with open('Assignment/Data-structures/sbp-dbp-reference-values-map.json', 'r') as file:
    sbp_map = json.load(file)

print(f'(Before removing incorret Male/Female SBP Values) {data.shape} ')

# Remover valores SBP invalidos para sexo Masculino
indices_to_drop = []
for index, row in data[data['Sexo'] == 'Masculino'].iterrows():
    idade = str(int(row['Idade']))
    sbp = row['SBP']
    lower_limit = float(sbp_map['M'][idade]['SBP'][0] - 10)
    upper_limit = float(sbp_map['M'][idade]['SBP'][1] + 10)
    if (sbp < lower_limit or sbp > upper_limit):
        indices_to_drop.append(index)
        
data = data.drop(indices_to_drop)
print(f'Male entries removed {len(indices_to_drop)}')

# Remover valores SBP invalidos para sexo Feminino
indices_to_drop = []
for index, row in data[data['Sexo'] == 'Feminino'].iterrows():
    idade = str(int(row['Idade']))
    sbp = row['SBP']
    lower_limit = float(sbp_map['F'][idade]['SBP'][0] - 10)
    upper_limit = float(sbp_map['F'][idade]['SBP'][1] + 10)
    if (sbp < lower_limit or sbp > upper_limit):
        indices_to_drop.append(index)
        
data = data.drop(indices_to_drop)

print(f'Female Entries removed {len(indices_to_drop)}')
print(f'(After removing incorret Male/Female SBP Values) {data.shape} ')

(Before removing incorret Male/Female SBP Values) (12384, 16) 
Male entries removed 206
Female Entries removed 143
(After removing incorret Male/Female SBP Values) (12035, 16) 


In [215]:
# Remover objectos com valores incorrectos de DBP (da-mos uma 'folga' de 10 para
# limite superior e inferior)
import json
with open('Assignment/Data-structures/sbp-dbp-reference-values-map.json', 'r') as file:
    dbp_map = json.load(file)

print(f'(Before removing incorret Male/Female DBP Values) {data.shape} ')

# Remover valores DBP invalidos para sexo Masculino
indices_to_drop = []
for index, row in data[data['Sexo'] == 'Masculino'].iterrows():
    idade = str(int(row['Idade']))
    dbp = row['DBP']
    lower_limit = float(dbp_map['M'][idade]['DBP'][0] - 10)
    upper_limit = float(dbp_map['M'][idade]['DBP'][1] + 10)
    if (dbp < lower_limit or dbp > upper_limit):
        indices_to_drop.append(index)
        
data = data.drop(indices_to_drop)
print(f'Male entries removed {len(indices_to_drop)}')

# Remover valores DBP invalidos para sexo Feminino
indices_to_drop = []
for index, row in data[data['Sexo'] == 'Feminino'].iterrows():
    idade = str(int(row['Idade']))
    dbp = row['DBP']
    lower_limit = float(dbp_map['F'][idade]['DBP'][0] - 10)
    upper_limit = float(dbp_map['F'][idade]['DBP'][1] + 10)
    if (dbp < lower_limit or dbp > upper_limit):
        indices_to_drop.append(index)
        
data = data.drop(indices_to_drop)

print(f'Female Entries removed {len(indices_to_drop)}')
print(f'(After removing incorret Male/Female DBP Values) {data.shape} ')

(Before removing incorret Male/Female DBP Values) (12035, 16) 
Male entries removed 19
Female Entries removed 29
(After removing incorret Male/Female DBP Values) (11987, 16) 


In [216]:
# Remover objectos com valores imposiveis de peso 
print(f'Before removing impossible Weight values {data.shape}')
data = data.drop(data[data['Peso'] <= 0].index)
print(f'After removing impossible Weight values {data.shape}')


Before removing impossible Weight values (11987, 16)
After removing impossible Weight values (10759, 16)


In [217]:
# Remover objectos com valores imposiveis de Altura 
print(f'Before removing impossible Heigth values {data.shape}')
data = data.drop(data[data['Altura'] <=0].index)
print(f'After removing impossible Heigth values {data.shape}')



Before removing impossible Heigth values (10759, 16)
After removing impossible Heigth values (9554, 16)


In [218]:
# Remover Pulsos = Outro
print(f'Before removing irrelevant Pulsos values {data.shape}')
data = data.drop(data[data['Pulsos'] == 'Outro'].index)
print(f'After removing irrelevant Pulsos values {data.shape}')


Before removing irrelevant Pulsos values (9554, 16)
After removing irrelevant Pulsos values (9541, 16)


> Removed objects with ```Pulsos = Outro```, this value gives no info about the Pulse related to the object, so we removed because we think this objects  will not help our prediction task

In [219]:
# Remover valores B2 = Outro
print(f'Before removing irrelevant B2 values {data.shape}')
data = data.drop(data[data['B2'] == 'Outro'].index)
print(f'After removing irrelevant B2 values {data.shape}')


Before removing irrelevant B2 values (9541, 16)
After removing irrelevant B2 values (9483, 16)


> Removed objects with ```B2 = Outro```, because this value gives no info about the type of the second heart sound related to the object, so we removed because we think this objects will not help our prediction task

In [220]:
# Remover valores Sopro = Sistolico e diastólico 
print(f'Before removing irrelevant Sopro values {data.shape}')
data = data.drop(data[data['Sopro'] == 'Sistolico e diastólico'].index)
print(f'After removing irrelevant Sopro values {data.shape}')

Before removing irrelevant Sopro values (9483, 16)
After removing irrelevant Sopro values (9482, 16)


> Removed objects with ```Sopro = Sistolico e diastólico```, because it just one object with this value, and we think it will not help our prediction task

In [221]:
# Ajustar unico valor HDA2 = Assintomático e passar para HDA1 = Assintomático
print(data['HDA2'].value_counts())

data.loc[data['HDA2'] == 'Assintomático', 'HDA1'] = 'Assintomático'
data.loc[data['HDA2'] == 'Assintomático', 'HDA2'] = np.nan
print(data['HDA2'].value_counts())

HDA2
Palpitacao         99
Dispneia           73
Dor precordial     67
Desmaio/tontura    48
Outro              36
Cianose            33
Ganho de peso      26
Assintomático       1
Name: count, dtype: int64
HDA2
Palpitacao         99
Dispneia           73
Dor precordial     67
Desmaio/tontura    48
Outro              36
Cianose            33
Ganho de peso      26
Name: count, dtype: int64


In [222]:
# Remover valores HDA1 = Outro e HDA2 = Outro
print(f'Before removing irrelevant HDA1 and HDA2 values {data.shape}')
data = data.drop(data[data['HDA1'] == 'Outro'].index)
data = data.drop(data[data['HDA2'] == 'Outro'].index)
print(f'After removing irrelevant HDA1 and HDA2 values {data.shape}')


Before removing irrelevant HDA1 and HDA2 values (9482, 16)
After removing irrelevant HDA1 and HDA2 values (9325, 16)


> Removed objects with ```HDA1 = Outro``` and ```HDA2 = Outro```, because the information about history of disease it provides is to ambiguos. Having this in mind, we think this objects will not help de prediction task, so we removed them

In [223]:
# Remover valores Sexo = Indeterminado 
print(f'Before removing irrelevant Sexo values {data.shape}')
data = data.drop(data[data['Sexo'] == 'Indeterminado'].index)
print(f'After removing irrelevant Sexo values {data.shape}')

Before removing irrelevant Sexo values (9325, 16)
After removing irrelevant Sexo values (9215, 16)


> Removed objects with ```Sexo = Indeterminado```, because it does not give any info about the patient sex, therefore it does not bring any value to the prediction task 

In [224]:
# Ajustar coluna FC para ser numérica 
# Tenta converter para número. O que tiver letras vira NaN (errors='coerce')
#data['FC'] = pd.to_numeric(data['FC'], errors='coerce')

In [225]:
# Remover objetos de Frequencia impossiveis
# import json
# with open('Assignment/Data-structures/fc-reference-values.json', 'r') as file:
#     fc_map = json.load(file)

# json.dumps(fc_map)

# print(f'(Before removing invalid FC Values) {data.shape} ')

# Remover valores SBP invalidos para sexo Feminino
# indices_to_drop = []
# for index, row in data.iterrows():
#     card_freq = row['FC']

#     idade = str(int(row['Idade']))
#     min_freq = fc_map[idade]['bpm_minimo'] - 5.0
#     max_freq = fc_map[idade]['bpm_maximo'] + 5.0
#     if (~np.isnan(card_freq)):
#         if(card_freq < min_freq or card_freq > max_freq):
#             indices_to_drop.append(index)

# data = data.drop(indices_to_drop)
# print(f'(After removing invalid FC Values) {data.shape} ')

### Address Missing Values

Deleting variables (columns) or objects (rows) that contain more than 50% missing values, provided the missing data has no inherent meaning. 

* Columns ```where a NaN has meaning```: 
    * **Sopro**: we assume it means there is no **murmur type** present (i.e. NaN in Sopro actualy means that is has values = **Ausente**)
    * **HDA1**, **HDA2** : we assume that there is no history of disease (i.e. NaN in HDA1 or HDA2 means that is value is Assintomático)
    * **Motivo1**, **Motivo2**: we assume that there is no motive (i.e. NaN in Motivo1 or Motivo2 means that is values is Outro )
 
* Columns ```where a NaN has no meaning```, and why we can remove: 
    * **Peso** : it means that the Weight wasnt measured, and therefore these objects don't help in our prediction task
    * **Altura** : it means that the Height wasnt measured, and therefore these objects don't help in our prediction task

    * **Pulsos** : it means that the Pulsos wasnt measured, and therefore these objects don't help in our prediction task
    * **SBP** and **DBP**: it means that the DBP and DBP werent measured, and therefore these objects don't help in our prediction task
    * **Patologia**: We cant be sure if NaN  means Anormal or Normal, so we the safest aproach is to assume it has no meaning
    * **B2**: it means that the type of the second heart sound wasnt measured, and therefore these objects don't help in our prediction task
    * **Sexo**: it means that Sex wasnt indicated, and therefore these objects don't help in our prediction task

* ```Important Note```:
    * **Result SBP-DBP**: No need to remove NaN because we are going to use the SBP and DBP values to recalculate all values of this column
    * **IMC** : No need to remove NaN because we are going to use the Peso and Altura values to recalculate all values of this column



In [226]:
# Columns that the NaN has no meaning and the respectiv object can be removed
columns_remove_NaN = ['Peso','Altura','Pulsos','SBP','DBP','Patologia','B2','Sexo']
print(f'(Before removing objects wiht missing Values with no meaning) {data.shape} ')
for col in columns_remove_NaN:
    data = data.dropna(subset=[col])
    print(f' Number of  objects(rows) after removing NaN from {col} : {len(data)} ')
print(f'(After removing objects wiht missing Values with no meaning) {data.shape} ')


(Before removing objects wiht missing Values with no meaning) (9215, 16) 
 Number of  objects(rows) after removing NaN from Peso : 9046 
 Number of  objects(rows) after removing NaN from Altura : 9046 
 Number of  objects(rows) after removing NaN from Pulsos : 8791 
 Number of  objects(rows) after removing NaN from SBP : 6820 
 Number of  objects(rows) after removing NaN from DBP : 6816 
 Number of  objects(rows) after removing NaN from Patologia : 6808 
 Number of  objects(rows) after removing NaN from B2 : 6807 
 Number of  objects(rows) after removing NaN from Sexo : 6806 
(After removing objects wiht missing Values with no meaning) (6806, 16) 


In [227]:
# Columns that the NaN has meaning, and the NaN need to be swicthed for a proper value
columns_alter_NaN = ['Sopro','HDA1','HDA2','Motivo1','Motivo2']
cor_NaN_value = ['Ausente','Assintomático','Assintomático','Outros','Outros']
print(f'(Before altering objects wiht missing Values with meaning) {data.shape} ')

i=0
for col in columns_alter_NaN:
    print(" (Before)" ,data[col].value_counts())
    data[col] = data[col].fillna(cor_NaN_value[i])
    print(" (After)" ,data[col].value_counts())
    i+=1


print(f'(After altering objects wiht missing Values with meaning) {data.shape} ')



(Before altering objects wiht missing Values with meaning) (6806, 16) 
 (Before) Sopro
Ausente       4714
Sistólico     2081
Contínuo         7
Diastólico       4
Name: count, dtype: int64
 (After) Sopro
Ausente       4714
Sistólico     2081
Contínuo         7
Diastólico       4
Name: count, dtype: int64
 (Before) HDA1
Assintomático      3785
Dor precordial      502
Dispneia            300
Palpitacao          266
Desmaio/tontura     160
Ganho de peso       131
Cianose              48
Name: count, dtype: int64
 (After) HDA1
Assintomático      5399
Dor precordial      502
Dispneia            300
Palpitacao          266
Desmaio/tontura     160
Ganho de peso       131
Cianose              48
Name: count, dtype: int64
 (Before) HDA2
Palpitacao         90
Dispneia           65
Dor precordial     55
Desmaio/tontura    40
Cianose            25
Ganho de peso      24
Name: count, dtype: int64
 (After) HDA2
Assintomático      6507
Palpitacao           90
Dispneia             65
Dor precordial    

In [230]:
missing_data = pd.DataFrame({
    'Coluna': data.columns,
    'Percentagem (%)': (data.isnull().mean() * 100).values
}).sort_values('Percentagem (%)', ascending=False)

print(missing_data)


            Coluna  Percentagem (%)
7   Result SBP-DBP         0.852189
0             Peso         0.000000
2              IMC         0.000000
1           Altura         0.000000
3            Idade         0.000000
4           Pulsos         0.000000
5              SBP         0.000000
6              DBP         0.000000
8        Patologia         0.000000
9               B2         0.000000
10           Sopro         0.000000
11            HDA1         0.000000
12            HDA2         0.000000
13            Sexo         0.000000
14         Motivo1         0.000000
15         Motivo2         0.000000


### Remove duplicates

In [235]:
print(f"Before removing duplicate lines {data.shape}")
data = data.drop_duplicates()
print(f"After removing duplicate lines {data.shape}")

Before removing duplicate lines (6806, 16)
After removing duplicate lines (6763, 16)


### Handling Outliers

In [ ]:
import math
# Selecionar colunas numéricas
cols = data.select_dtypes(include=['number']).columns
n_cols = len(cols)

# Calcular número de linhas para os subplots (ex: 3 colunas por linha)
cols_per_row = 3
rows = math.ceil(n_cols / cols_per_row)

# Configurar a área de desenho
fig, axes = plt.subplots(rows, cols_per_row, figsize=(15, rows * 4))
axes = axes.flatten()

for i, col in enumerate(cols):
    sns.boxplot(y=data[col], ax=axes[i])
    axes[i].set_title(f'Outliers em {col}')
    axes[i].set_ylabel('')

# Remover eixos que sobrarem (se houver)
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()

### TODO

1) Colocar os valores de SBP e DBP como Normal, Limite e Hipertenso (de acordo com os valores de referencia)
2) Recalcular Result SBP-PPA
3) Recalcular BMI (remover os passos do valor de refrencia)
4) Alterar Idade para ser categorica (baseado no intervalo)
  

> Usar o document.pdf como referência

## Data Preprocessing